Tiktokenizer - https://tiktokenizer.vercel.app/?model=gpt2

here you visualize tokenization of text


<img src="public/tiktokenizer.png" width="500">

In [402]:
# download text in file
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    response = requests.get(url)
    with open(file_path, "wb") as f:
        f.write(response.content)

In [403]:
# load text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [404]:
len(raw_text)

20479

In [405]:
print(raw_text[:20])
print("...")
print(raw_text[-20:])

I HAD always thought
...
ng our kind of art."


In [406]:
# split text into tokens
import re

text = "Hello, world. This, is a test."
result = re.split(r"(\s)", text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [407]:
# comma and period as separate token
result = re.split(r"([,.]|\s)", text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [408]:
# remove whitespaces
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [409]:
# more symbols as separate tokens
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [410]:
# raw_text into tokens
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocesssed = result

In [411]:
print(preprocesssed[:10])
print("...")
print(preprocesssed[-10:])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']
...
["'", 's', 'no', 'exterminating', 'our', 'kind', 'of', 'art', '.', '"']


In [412]:
len(preprocesssed)

4690

In [413]:
all_words = sorted(set(preprocesssed))
print(all_words[:10])
print("...")
print(all_words[-10:])

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';']
...
['would', 'wouldn', 'year', 'years', 'yellow', 'yet', 'you', 'younger', 'your', 'yourself']


In [414]:
vocab_size = len(all_words)
vocab_size

1130

In [415]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [416]:
items = list(vocab.items())
print(items[:5])
print("...")
print(items[-5:])

[('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4)]
...
[('yet', 1125), ('you', 1126), ('younger', 1127), ('your', 1128), ('yourself', 1129)]


In [417]:
from typing import Any


class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str | Any, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return text

In [418]:
tokenizer = SimpleTokenizerV1(vocab)
tokenizer.decode(tokenizer.encode("What are you doing?"))

'What are you doing ?'

Note that in above example there's a space before question mark which is undesired

In [419]:
# Replace spaces before the specified punctuations
class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str | Any, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r"\1", text)
        return text

In [420]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [421]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [422]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [423]:
tokenizer.decode(tokenizer.encode("What are you doing?"))

'What are you doing?'

In [424]:
# note that tokenizer will only work for words present in raw_text based on which vocabulary is built
try:
    text1 = "Hello, do you like tea?"
    ids = tokenizer.encode(text1)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

KeyError: 'Hello'


In [425]:
# adding special tokens
# handling unknown vocab (though this method leads to loss of information)
# it's better to split unknown words into character tokens
all_tokens = sorted(set(preprocesssed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [426]:
len(vocab.items())

1132

In [427]:
for item in list(vocab.items())[-5:]:
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [428]:
class SimpleTokenizerV2:
    def __init__(self, vocab: dict[str | Any, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r"\1", text)
        return text

In [429]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [430]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [431]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'